# Digital Twin app — Colab launcher

Run the four cells in order. The last one prints a **public URL**; everything else
(consent, interview, voice and photo upload, building the twin, chatting) happens in that page.

**Runtime → Change runtime type → T4 GPU** first.

After a disconnect, run all four cells again. Your answers, voice, photo, corrections
and notes come back from Google Drive, and the twin rebuilds itself.

## 1. Connect Drive and get the code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, shutil
DRIVE = '/content/drive/MyDrive/twin_assets'
os.makedirs(DRIVE, exist_ok=True)
saved = os.path.join(DRIVE, 'digital-twin.zip')
if not os.path.exists(saved):
    from google.colab import files
    print('Pick the project zip (digital-twin-app.zip). This happens once; it is kept on Drive.')
    up = list(files.upload().keys())
    shutil.copy(next(f for f in up if f.endswith('.zip')), saved)
!rm -rf /content/digital-twin && mkdir -p /content/digital-twin && unzip -q -o "{saved}" -d /content/digital-twin
!ls /content/digital-twin

## 2. Install the libraries  *(about 4 minutes)*

In [ ]:
!pip -q install fastapi "uvicorn[standard]" python-multipart faiss-cpu sentence-transformers pyyaml \
    accelerate bitsandbytes "transformers>=4.57,<5" coqui-tts
print('done')

## 3. Voice and avatar models  *(about 15 minutes, optional)*

Skip this cell for a text-only twin. The app still runs; the voice and avatar switches stay off.

In [ ]:
!bash /content/digital-twin/src/setup_avatar.sh
!bash /content/digital-twin/src/setup_wav2lip.sh
!cp /content/digital-twin/src/enhance_frames.py /content/enh.py
print('avatar ready')

## 4. Start the app

Loads Phi-3-mini in the background and prints the public URL. Open it on any device.
The URL changes each time this cell runs.

In [ ]:
import subprocess, time, re, os
subprocess.run('pkill -f "python -m app"', shell=True)
env = dict(os.environ, COQUI_TOS_AGREED='1', MPLBACKEND='Agg')
log = open('/content/app.log', 'w')
proc = subprocess.Popen(['python', '-m', 'app', '--share', '--work', '/content', '--drive', DRIVE],
                        cwd='/content/digital-twin', stdout=log, stderr=subprocess.STDOUT, env=env)
url = None
for _ in range(120):
    time.sleep(2)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/app.log').read())
    if m: url = m.group(0); break
    if proc.poll() is not None: break
print(open('/content/app.log').read()[-1500:])
print('\nOpen this link:', url or 'not ready, run this cell again or check /content/app.log')

### Notes
- The page works while the model is still loading; the chip in the top right says when Phi-3 is ready.
- Leave this tab open. Closing it ends the Colab session and the link stops working.
- The full server log is in `/content/app.log`.